In [1]:
# !nvidia-smi

In [2]:
from google.colab import userdata
import os
import sys
import time
import pandas as pd
import subprocess
from tqdm.auto import tqdm

In [3]:
token = userdata.get("GITHUB_TOKEN")
repo_path = "/content/hw1_task1"

repo_url = ("https://x-access-token:" + token + "@github.com/kryalka/hw1_task1.git")

git_folder = repo_path + "/.git"
repo_already_exists = os.path.isdir(git_folder)

if repo_already_exists:
    subprocess.run(
        [
            "git",
            "-C",
            repo_path,
            "pull",
            repo_url,
            "main",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            repo_url,
            repo_path,
        ],
        check=True,
    )

del token
del repo_url


In [4]:
%cd /content/hw1_task1

%env PYTHONPATH=/content/hw1_task1

/content/hw1_task1
env: PYTHONPATH=/content/hw1_task1


In [5]:
%pip install -q hypothesis pytest pytest-env colorama typing_extensions

In [6]:
# import minitorch

# print(minitorch.__file__)

In [7]:
!pytest tests/ -m "task3_3 or task3_4" -q

................................................................         [100%]
=============================== warnings summary ===============================
tests/test_tensor_general.py: 20 warnings
  /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
    warn(errors.NumbaPerformanceWarning(msg))

tests/test_tensor_general.py: 78 warnings
  /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
    warn(NumbaPerformanceWarning(msg))

tests/test_tensor_general.py: 12 warnings
  /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
    warn(errors.NumbaPerformanceWarning(msg))

tests/test_tensor_

3.5

In [ ]:
def train(dataset, backend):
    epochs = 500

    logs_folder = "/content/training_logs"
    os.makedirs(logs_folder, exist_ok=True)

    log_path = f"{logs_folder}/{dataset}_{backend}.txt"

    command = [
        sys.executable,
        "-u",
        "project/run_fast_tensor.py",
        "--BACKEND",
        backend,
        "--HIDDEN",
        "100",
        "--DATASET",
        dataset,
        "--RATE",
        "0.05",
        "--PTS",
        "50",
    ]

    # start = time.time()
    # сделала чуть по-другому: первую эпоху не учитываю,
    # чтобы время получалось точнее

    cold_start = time.time()
    warm_start = None

    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    progress = tqdm(total=epochs, desc=f"{dataset} {backend}")

    with open(log_path, "w") as log_file:
        for line in process.stdout:
            log_file.write(line)

            if line.startswith("Epoch"):
                parts = line.split()
                epoch = int(parts[1])
                completed_epochs = epoch + 1

                progress.update(completed_epochs - progress.n)

                if epoch == 0 and warm_start is None:
                    warm_start = time.time()

    return_code = process.wait()
    end_time = time.time()

    progress.update(epochs - progress.n)
    progress.close()

    cold_total_time = end_time - cold_start
    cold_time_per_epoch = cold_total_time / epochs

    if warm_start is not None:
        warm_epochs = epochs - 1
        warm_total_time = end_time - warm_start
        warm_time_per_epoch = warm_total_time / warm_epochs
    else:
        warm_time_per_epoch = None

    print(f"{dataset} {backend}")
    print(f"Cold time per epoch: {cold_time_per_epoch:.3f}s")

    if warm_time_per_epoch is not None:
        print(f"Warm time per epoch: {warm_time_per_epoch:.3f}s")

In [9]:
train("simple", "gpu")

simple gpu:   0%|          | 0/500 [00:00<?, ?it/s]

simple gpu time per epoch: 1.545s


In [10]:
train("split", "gpu")

split gpu:   0%|          | 0/500 [00:00<?, ?it/s]

split gpu time per epoch: 1.538s


In [11]:
train("xor", "gpu")

xor gpu:   0%|          | 0/500 [00:00<?, ?it/s]

xor gpu time per epoch: 1.553s


In [12]:
train("split", "cpu")

split cpu:   0%|          | 0/500 [00:00<?, ?it/s]

split cpu time per epoch: 0.146s
